# 24 · Tree models vs the reference Ridge

## Objective

Feature engineering is finished (notebook 23). This notebook compares three models on **exactly the same data, features,
folds and metrics**:

1. **Ridge**, the reference of notebook 23 (MAE ₹69.37 lakh, Median APE 21.37%, RMSE ₹177.69 lakh, R² 0.491);
2. **Random Forest**, in the same fixed configuration as notebook 20;
3. **HistGradientBoostingRegressor** (HGB), scikit-learn's gradient-boosting model, in a fixed default configuration.

No tuning, no new features, no locality target encoding, and the test set is not used.

**The question from notebook 20.** With the old features (centre latitude/longitude), the Random Forest was better than
Ridge for **typical** listings (MAE ₹68.6 vs ₹70.2 lakh, Median APE 20.2% vs 21.8%), but it **under-predicted expensive
homes badly** (₹5 Cr+: median −50% vs −35% for Ridge), so its RMSE (₹199.4 lakh) and R² (0.36) were clearly worse. We want
to see whether the distance features change this, and whether boosting behaves differently from a forest.

| Fixed setting | Value |
|---|---|
| Rows | the 16,219 training listings; the 4,081 test listings are removed from memory |
| Features | `build_model_features()`: 19 notebook-18 features + 6 locality-centre distances (notebook 23) |
| Folds | `GroupKFold(n_splits=5, shuffle=True, random_state=42)`, `groups = property_group_id` |
| Target | every model learns `log(price)` (`TransformedTargetRegressor`); predictions are converted back to rupees |
| Metrics | MAE, Median APE, RMSE, R², mean ± sample std over the folds |

## 1. Load data, lock the test set, build the features

In [1]:
from pathlib import Path
import hashlib
import sys
import time

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEANED_CSV = PROJECT_ROOT / "data" / "processed" / "cleaned_listings.csv"
SPLIT_CSV = PROJECT_ROOT / "data" / "processed" / "train_test_split.csv"
LOCALITIES_CSV = PROJECT_ROOT / "data" / "extracted" / "localities.csv"
FEATURES_PY = PROJECT_ROOT / "src" / "features" / "build_features.py"

sys.path.insert(0, str(PROJECT_ROOT))
from src.features.build_features import DISTANCE_FEATURES, INPUT_COLUMNS, build_model_features


def file_md5(path):
    # A fingerprint of the file's bytes. If the file changes, the fingerprint changes.
    return hashlib.md5(path.read_bytes()).hexdigest()


# The same source files as notebooks 19-23
EXPECTED_MD5 = {
    CLEANED_CSV: "3a81a385a9aca28d5b70322e1e143c08",
    SPLIT_CSV: "10693e86d1d8a55e606399d718a8f556",
    LOCALITIES_CSV: "8d9bb54dfc2fb49673e74b976ab73ace",
    FEATURES_PY: "4fb50b05aa072d1a9ae25851dbf1cc69",   # the version checked in notebook 23
}
for path, md5 in EXPECTED_MD5.items():
    assert file_md5(path) == md5, f"{path.name} is not the file used by notebook 23"
    print(f"{path.name:28} {md5}")

cleaned_listings.csv         3a81a385a9aca28d5b70322e1e143c08
train_test_split.csv         10693e86d1d8a55e606399d718a8f556
localities.csv               8d9bb54dfc2fb49673e74b976ab73ace
build_features.py            4fb50b05aa072d1a9ae25851dbf1cc69


In [2]:
listings = pd.read_csv(CLEANED_CSV, low_memory=False)
split_assignment = pd.read_csv(SPLIT_CSV)
locality_table = pd.read_csv(LOCALITIES_CSV, usecols=["locality_id", "locality_name", "locality_lat", "locality_long"])

listings = listings.merge(split_assignment[["property_id", "split"]], on="property_id", validate="one_to_one")
assert len(listings) == 20300
assert (listings["split"] == "train").sum() == 16219
assert (listings["split"] == "test").sum() == 4081
train_groups = set(listings.loc[listings["split"] == "train", "property_group_id"])
test_groups = set(listings.loc[listings["split"] == "test", "property_group_id"])
assert train_groups.isdisjoint(test_groups), "A property group is on both sides!"

train = listings[listings["split"] == "train"].copy()
test_property_ids = set(listings.loc[listings["split"] == "test", "property_id"])
test_row_count = len(test_property_ids)
del listings   # the test rows are gone from memory from here on

model_features = build_model_features(train, locality_table)
y_train = train["price"]
groups = train["property_group_id"]

RANDOM_STATE = 42
cv_splits = list(GroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
                 .split(model_features, y_train, groups=groups))
expected_sizes = [(12997, 3222), (12980, 3239), (12961, 3258), (12989, 3230), (12949, 3270)]
for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
    assert (len(train_positions), len(validation_positions)) == expected_sizes[fold_number - 1]

print("Training rows:", len(train), "| test rows locked:", test_row_count)
print("Feature table:", model_features.shape, "| fold sizes identical to notebooks 19-23")

Training rows: 16219 | test rows locked: 4081
Feature table: (16219, 25) | fold sizes identical to notebooks 19-23


## 2. Preprocessing: one design matrix for all three models

**Decision:** all three models receive **the same preprocessed columns**, produced by the notebook-23
`ColumnTransformer` fitted inside each fold:

| Group | Columns | Steps (learned from each fold's training part) |
|---|---|---|
| numeric (11) | bedrooms, bathrooms, log_area, apartment_floor, apartment_building_floors, 6 distances | median imputation → `StandardScaler` |
| binary (5) | area_missing, area_unit_is_sqyd, has_land_area_unit, prime_location_flag, in_named_project | passthrough |
| categorical (8) | property_type, balconies_band, transaction_group, furnishing, possession_status, age_of_property, facing_clean, posted_by | one-hot |
| locality (1) | locality_code | one-hot; < 10 listings in the fold's training part → one "rare" column |

**Why this matters for HistGradientBoosting.** HGB has two optional abilities that Ridge and the Random Forest do not use
here:

- **native categorical support** (`categorical_features=...` on ordinal-encoded columns): the trees can split a
  category column into any two groups of categories;
- **native missing-value handling**: HGB can learn which side of a split missing values should go to.

Using them would change **how** HGB sees the information (and it limits a categorical column to at most 255 categories,
so the ≈ 1,800 localities would still need grouping). That would mix two effects: the model and its preprocessing. For
a clean first comparison every model gets the same matrix. The difference in results then comes from the **model
alone**, and no model gets information another does not have. Missing values are covered for every model in the same way:
median imputation, plus the existing `area_missing` flag and the "unknown" categories.

Scaling does not change a tree (a tree only uses the order of values), so the scaler is harmless for the tree models.
HGB's native categorical and missing-value handling are **possible options for the tuning notebook**. Any gain from them
would have to be shown on the same folds.

The one-hot output is **dense** (`sparse_output=False`, as in notebooks 19–23). HGB needs dense input, and about 300
columns × 13,000 rows is small.

In [3]:
NUMERIC_FEATURES = ["bedrooms", "bathrooms", "log_area", "apartment_floor", "apartment_building_floors"] + DISTANCE_FEATURES
BINARY_FEATURES = ["area_missing", "area_unit_is_sqyd", "has_land_area_unit", "prime_location_flag", "in_named_project"]
CATEGORICAL_FEATURES = ["property_type", "balconies_band", "transaction_group", "furnishing", "possession_status",
                        "age_of_property", "facing_clean", "posted_by"]
LOCALITY_FEATURES = ["locality_code"]
assert sorted(NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES + LOCALITY_FEATURES) == sorted(model_features.columns)


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
             NUMERIC_FEATURES),
            ("binary", "passthrough", BINARY_FEATURES),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
            ("locality", OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist", sparse_output=False),
             LOCALITY_FEATURES),
        ]
    )


def make_log_price_model(regressor):
    # preprocessing + a regressor that learns log(price); predictions come back in rupees
    return Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", TransformedTargetRegressor(regressor=regressor, func=np.log, inverse_func=np.exp)),
    ])

## 3. The three models (fixed configurations, not tuned)

| Model | Settings | Why |
|---|---|---|
| **Ridge** | `alpha=1.0` | the notebook-19/23 reference, unchanged |
| **Random Forest** | `n_estimators=300`, `max_features="sqrt"`, `min_samples_leaf=2`, `random_state=42`, `n_jobs=-1` | exactly notebook 20's configuration, so the only change from notebook 20 is the feature set |
| **HistGradientBoosting** | scikit-learn defaults (`learning_rate=0.1`, `max_iter=100`, `max_leaf_nodes=31`, `min_samples_leaf=20`, `l2_regularization=0`), **`early_stopping=False`**, `random_state=42` | a standard untuned starting point |

**How gradient boosting differs from a Random Forest.** A forest builds many deep trees **independently** and
**averages** them. Boosting builds small trees **one after another**: each new tree is fitted to the errors the previous
trees still make, and its prediction is added with a small weight (`learning_rate`). Averaging keeps a forest's
predictions inside the range of each leaf's training prices. Boosting adds corrections step by step, which can push
predictions further toward the extremes.

**Why `early_stopping=False`.** By default HGB turns on early stopping for more than 10,000 rows. It then holds back a
**random** 10% of the fold's training part to decide when to stop. That internal split is **not group-aware**, so copies
of the same property could sit on both sides and make it stop at the wrong time. Turning it off gives a fixed 100
boosting rounds: simple, reproducible, and no hidden split.

In [4]:
MODELS = {
    "Ridge": make_log_price_model(Ridge(alpha=1.0)),
    "Random Forest": make_log_price_model(RandomForestRegressor(n_estimators=300, max_features="sqrt",
                                                                min_samples_leaf=2, random_state=RANDOM_STATE,
                                                                n_jobs=-1)),
    "HistGradientBoosting": make_log_price_model(HistGradientBoostingRegressor(early_stopping=False,
                                                                               random_state=RANDOM_STATE)),
}
hgb_settings = MODELS["HistGradientBoosting"].named_steps["model"].regressor.get_params()
print({key: hgb_settings[key] for key in ["learning_rate", "max_iter", "max_leaf_nodes", "min_samples_leaf",
                                          "l2_regularization", "early_stopping", "loss"]})

{'learning_rate': 0.1, 'max_iter': 100, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 0.0, 'early_stopping': False, 'loss': 'squared_error'}


## 4. Cross-validation

The fold loop of notebooks 20–23: for each fold, `clone()` makes a fresh, unfitted copy of the whole pipeline, fits it
on the fold's training part, and predicts the validation part. All three models loop over **the same `cv_splits` list**.

In [5]:
def median_ape(actual, predicted):
    return np.median(np.abs(predicted - actual) / actual) * 100


def run_cv(model):
    fold_rows = []
    out_of_fold = pd.Series(np.nan, index=model_features.index)
    fold_checks = []
    for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
        fold_model = clone(model)
        fold_model.fit(model_features.iloc[train_positions], y_train.iloc[train_positions])
        actual = y_train.iloc[validation_positions]
        predicted = fold_model.predict(model_features.iloc[validation_positions])
        out_of_fold.iloc[validation_positions] = predicted
        fold_rows.append({"fold": fold_number,
                          "MAE": mean_absolute_error(actual, predicted),
                          "RMSE": root_mean_squared_error(actual, predicted),
                          "R2": r2_score(actual, predicted),
                          "Median APE": median_ape(actual, predicted)})
        preprocess = fold_model.named_steps["preprocess"]
        fold_checks.append({
            "train_positions": train_positions,
            "validation_positions": validation_positions,
            "imputer_medians": preprocess.named_transformers_["numeric"].named_steps["impute"].statistics_.copy(),
            "expected_medians": model_features.iloc[train_positions][NUMERIC_FEATURES].median().values,
            "rare_localities": len(preprocess.named_transformers_["locality"].infrequent_categories_[0]),
            "prediction_count": len(predicted),
        })
    return pd.DataFrame(fold_rows), out_of_fold, fold_checks


def summarise(model_name, fold_table):
    return {
        "Model": model_name,
        "MAE (₹ lakh)": fold_table["MAE"].mean() / 1e5,
        "MAE std": fold_table["MAE"].std() / 1e5,
        "Median APE (%)": fold_table["Median APE"].mean(),
        "Median APE std": fold_table["Median APE"].std(),
        "RMSE (₹ lakh)": fold_table["RMSE"].mean() / 1e5,
        "RMSE std": fold_table["RMSE"].std() / 1e5,
        "R²": fold_table["R2"].mean(),
        "R² std": fold_table["R2"].std(),
    }


fold_tables, out_of_fold_predictions, fold_checks = {}, {}, {}
for name, model in MODELS.items():
    start = time.time()
    fold_tables[name], out_of_fold_predictions[name], fold_checks[name] = run_cv(model)
    print(f"{name:22} done in {time.time() - start:5.1f} s")

Ridge                  done in   3.6 s
Random Forest          done in  16.8 s
HistGradientBoosting   done in  48.2 s


## 5. Overall results

In [6]:
ridge = fold_tables["Ridge"]
# The Ridge reference must reproduce notebook 23 exactly
assert np.isclose(ridge["MAE"].mean() / 1e5, 69.365, atol=0.001)
assert np.isclose(ridge["Median APE"].mean(), 21.372, atol=0.001)
assert np.isclose(ridge["RMSE"].mean() / 1e5, 177.689, atol=0.001)
assert np.isclose(ridge["R2"].mean(), 0.491, atol=0.001)
assert np.allclose(ridge["MAE"].values / 1e5, [66.365, 65.912, 72.874, 69.542, 72.134], atol=0.001)
print("Ridge reproduces notebook 23 exactly (means and every fold's MAE).")
print()
summary = pd.DataFrame([summarise(name, fold_tables[name]) for name in MODELS])
summary.round(3)

Ridge reproduces notebook 23 exactly (means and every fold's MAE).



,Model,MAE (₹ lakh),MAE std,Median APE (%),Median APE std,RMSE (₹ lakh),RMSE std,R²,R² std
0,Ridge,69.365,3.199,21.372,0.430,177.689,20.734,0.491,0.029
1,Random Forest,66.221,3.681,19.520,0.617,193.025,27.118,0.402,0.052
2,HistGradientBoosting,58.354,2.832,17.041,0.415,164.759,15.766,0.561,0.026


### Per fold, and how often each tree model beats Ridge

In [7]:
for metric, scale, label in [("MAE", 1e5, "MAE (₹ lakh)"), ("Median APE", 1, "Median APE (%)"),
                             ("RMSE", 1e5, "RMSE (₹ lakh)"), ("R2", 1, "R²")]:
    table = pd.DataFrame({name: fold_tables[name][metric].values / scale for name in MODELS},
                         index=pd.Index(range(1, 6), name="fold"))
    print(label)
    print(table.round(3).to_string())
    print()

win_rows = []
for name in ["Random Forest", "HistGradientBoosting"]:
    table = fold_tables[name]
    win_rows.append({
        "Model": name,
        "folds with lower MAE than Ridge": (table["MAE"] < ridge["MAE"]).sum(),
        "folds with lower Median APE": (table["Median APE"] < ridge["Median APE"]).sum(),
        "folds with lower RMSE": (table["RMSE"] < ridge["RMSE"]).sum(),
        "folds with higher R²": (table["R2"] > ridge["R2"]).sum(),
    })
pd.DataFrame(win_rows).set_index("Model")

MAE (₹ lakh)
       Ridge  Random Forest  HistGradientBoosting
fold                                             
1     66.365         62.149                55.363
2     65.912         64.152                56.168
3     72.874         70.734                62.269
4     69.542         64.620                57.928
5     72.134         69.449                60.040

Median APE (%)
       Ridge  Random Forest  HistGradientBoosting
fold                                             
1     21.002         18.597                16.311
2     20.843         19.899                17.143
3     21.482         19.945                17.348
4     21.761         19.167                17.210
5     21.771         19.994                17.193

RMSE (₹ lakh)
        Ridge  Random Forest  HistGradientBoosting
fold                                              
1     147.647        155.134               140.560
2     175.546        192.840               165.531
3     179.474        192.758               169.627
4

,folds with lower MAE than Ridge,folds with lower Median APE,folds with lower RMSE,folds with higher R²
Model,,,,
Random Forest,5,5,0,0
HistGradientBoosting,5,5,5,5


### Did the distance features change the Random Forest's behaviour?

Notebook 20 ran **the same Random Forest configuration on the same folds**, with the old features (centre lat/long
instead of distances). Its results are quoted below from notebook 20's output (not re-run here), next to this notebook's
Random Forest and Ridge. The only difference between the two Random Forest rows is the feature set.

In [8]:
NOTEBOOK_20 = {   # printed by notebook 20 (same folds, old features)
    "Ridge, lat/long (nb 19/20)": {"MAE (₹ lakh)": 70.248, "Median APE (%)": 21.841, "RMSE (₹ lakh)": 179.528, "R²": 0.481},
    "Random Forest, lat/long (nb 20)": {"MAE (₹ lakh)": 68.580, "Median APE (%)": 20.237, "RMSE (₹ lakh)": 199.358,
                                        "R²": 0.362},
}
feature_effect = pd.DataFrame(NOTEBOOK_20).T
for name, label in [("Ridge", "Ridge, distances (this notebook)"), ("Random Forest", "Random Forest, distances (this notebook)"),
                    ("HistGradientBoosting", "HistGradientBoosting, distances (this notebook)")]:
    row = summarise(name, fold_tables[name])
    feature_effect.loc[label] = [row["MAE (₹ lakh)"], row["Median APE (%)"], row["RMSE (₹ lakh)"], row["R²"]]
feature_effect.round(3)

,MAE (₹ lakh),Median APE (%),RMSE (₹ lakh),R²
"Ridge, lat/long (nb 19/20)",70.248,21.841,179.528,0.481
"Random Forest, lat/long (nb 20)",68.580,20.237,199.358,0.362
"Ridge, distances (this notebook)",69.365,21.372,177.689,0.491
"Random Forest, distances (this notebook)",66.221,19.520,193.025,0.402
"HistGradientBoosting, distances (this notebook)",58.354,17.041,164.759,0.561


## 6. Where the models differ

Out-of-fold predictions: every training listing is predicted by the model of the fold in which it was validated. For
each group: MAE (₹ lakh), Median APE (%) and the **median signed % error** (+ = predicts too high, − = too low).
`price_band` is derived from the target and is used **only for reporting**, never as a feature.

In [9]:
RANGE_NAMES = ["0-50 L", "50-80 L", "80 L-1.2 Cr", "1.2-1.5 Cr", "1.5-1.75 Cr",
               "1.75-2 Cr", "2-3 Cr", "3-5 Cr", "5 Cr+"]
PROPERTY_TYPES = ["Apartment", "Builder Floor Apartment", "Penthouse", "Residential House", "Villa"]
SHORT = {"Ridge": "Ridge", "Random Forest": "RF", "HistGradientBoosting": "HGB"}

errors = pd.DataFrame({
    "price": y_train,
    "price_band": train["price_band"],
    "property_type": train["property_type"],
    "type_family": np.where(train["property_type"].isin(["Residential House", "Villa"]), "house / villa",
                            "apartment-like"),
}, index=train.index)
for name in MODELS:
    errors[SHORT[name] + " abs error"] = (out_of_fold_predictions[name] - y_train).abs()
    errors[SHORT[name] + " abs % error"] = errors[SHORT[name] + " abs error"] / y_train * 100
    errors[SHORT[name] + " signed % error"] = (out_of_fold_predictions[name] - y_train) / y_train * 100


def compare_by(group_column, order):
    grouped = errors.groupby(group_column)
    table = pd.DataFrame({"listings": grouped.size()})
    for name in MODELS:
        table[SHORT[name] + " MAE"] = grouped[SHORT[name] + " abs error"].mean() / 1e5
    for name in MODELS:
        table[SHORT[name] + " APE"] = grouped[SHORT[name] + " abs % error"].median()
    for name in MODELS:
        table[SHORT[name] + " signed"] = grouped[SHORT[name] + " signed % error"].median()
    return table.reindex(order).round(1)


print("MAE in ₹ lakh; APE = median absolute % error; signed = median signed % error (+ too high, - too low)")

MAE in ₹ lakh; APE = median absolute % error; signed = median signed % error (+ too high, - too low)


### By price band

In [10]:
compare_by("price_band", RANGE_NAMES)

,listings,Ridge MAE,RF MAE,HGB MAE,Ridge APE,RF APE,HGB APE,Ridge signed,RF signed,HGB signed
price_band,,,,,,,,,,
0-50 L,1260,26.4,26.9,18.1,50.2,53.8,26.9,50.0,53.8,25.6
50-80 L,2291,20.8,20.5,18.4,19.9,19.0,16.5,16.0,16.5,8.0
80 L-1.2 Cr,2402,23.7,24.9,23.8,15.4,17.5,16.2,4.5,11.6,7.4
1.2-1.5 Cr,2287,34.3,28.9,29.6,17.5,14.9,14.8,-0.7,4.9,3.9
1.5-1.75 Cr,1687,41.2,30.2,30.8,18.7,13.7,12.8,-2.3,0.0,-0.0
1.75-2 Cr,1616,49.2,34.3,38.1,19.0,13.7,13.6,-4.2,-3.4,-2.6
2-3 Cr,2162,70.3,50.2,53.7,20.8,16.9,16.0,-9.1,-12.5,-8.4
3-5 Cr,1550,130.7,111.9,105.6,29.1,26.6,22.8,-18.3,-24.8,-14.6
5 Cr+,964,420.5,497.1,376.9,36.3,46.5,30.8,-34.5,-46.5,-27.9


### By property type

In [11]:
compare_by("property_type", PROPERTY_TYPES)

,listings,Ridge MAE,RF MAE,HGB MAE,Ridge APE,RF APE,HGB APE,Ridge signed,RF signed,HGB signed
property_type,,,,,,,,,,
Apartment,9218,38.8,36.7,32.0,17.3,15.4,13.3,0.5,1.2,0.4
Builder Floor Apartment,498,33.7,35.4,30.6,22.3,23.3,19.0,1.0,9.3,2.0
Penthouse,34,44.3,59.7,44.5,18.1,45.8,28.2,3.7,40.1,6.0
Residential House,4353,98.5,90.1,82.7,31.3,27.3,25.2,0.6,1.5,0.8
Villa,2116,151.5,153.1,130.2,27.9,25.1,22.4,3.8,-3.0,1.4


### Apartments vs houses/villas

In [12]:
compare_by("type_family", ["apartment-like", "house / villa"])

,listings,Ridge MAE,RF MAE,HGB MAE,Ridge APE,RF APE,HGB APE,Ridge signed,RF signed,HGB signed
type_family,,,,,,,,,,
apartment-like,9750,38.6,36.7,31.9,17.5,15.7,13.6,0.5,1.6,0.5
house / villa,6469,115.8,110.7,98.2,30.2,26.7,24.3,1.8,-0.3,0.9


### The expensive end

Two views of the problem notebook 20 found:

- the **₹20 Cr+** listings (the luxury threshold of `docs/outlier_handling.md`, B2), as in notebook 20;
- **how much of each model's squared error comes from the ₹5 Cr+ band**. RMSE and R² are built from squared errors, so
  if one band dominates them, those metrics mostly describe that band.

In [13]:
LUXURY_THRESHOLD = 20e7   # ₹20 crore (docs/outlier_handling.md, B2)
is_luxury = errors["price"] >= LUXURY_THRESHOLD
is_top_band = errors["price_band"] == "5 Cr+"

rows = []
for name in MODELS:
    squared_error = (out_of_fold_predictions[name] - y_train) ** 2
    rows.append({
        "Model": name,
        "₹20 Cr+ listings": is_luxury.sum(),
        "₹20 Cr+ MAE (₹ lakh)": errors.loc[is_luxury, SHORT[name] + " abs error"].mean() / 1e5,
        "₹20 Cr+ median signed %": errors.loc[is_luxury, SHORT[name] + " signed % error"].median(),
        "share of squared error from ₹5 Cr+ (%)": squared_error[is_top_band].sum() / squared_error.sum() * 100,
        "RMSE without ₹5 Cr+ (₹ lakh)": np.sqrt(squared_error[~is_top_band].mean()) / 1e5,
    })
print("Notebook 20 (old features): ₹20 Cr+ median signed error -70.6% (Ridge), -84.3% (Random Forest)")
pd.DataFrame(rows).set_index("Model").round(1)

Notebook 20 (old features): ₹20 Cr+ median signed error -70.6% (Ridge), -84.3% (Random Forest)


,₹20 Cr+ listings,₹20 Cr+ MAE (₹ lakh),₹20 Cr+ median signed %,share of squared error from ₹5 Cr+ (%),RMSE without ₹5 Cr+ (₹ lakh)
Model,,,,,
Ridge,48,1968.4,-70.5,82.1,78.0
Random Forest,48,2377.1,-80.9,91.0,60.2
HistGradientBoosting,48,1889.7,-64.1,86.0,63.7


In [14]:
# How far up can each model predict? The largest out-of-fold prediction vs the largest price, and the share of
# ₹5 Cr+ listings predicted below ₹5 Cr
rows = []
for name in MODELS:
    predictions = out_of_fold_predictions[name]
    rows.append({
        "Model": name,
        "largest prediction (₹ lakh)": predictions.max() / 1e5,
        "₹5 Cr+ listings predicted below ₹5 Cr (%)": (predictions[is_top_band] < 5e7).mean() * 100,
        "₹0-50 L listings predicted above ₹50 L (%)": (predictions[errors["price_band"] == "0-50 L"] > 5e6).mean() * 100,
    })
print("Largest training price: ₹", round(y_train.max() / 1e5, 1), "lakh")
pd.DataFrame(rows).set_index("Model").round(1)

Largest training price: ₹ 5500.0 lakh


,largest prediction (₹ lakh),₹5 Cr+ listings predicted below ₹5 Cr (%),₹0-50 L listings predicted above ₹50 L (%)
Model,,,
Ridge,3580.6,53.9,70.6
Random Forest,1108.4,72.6,80.1
HistGradientBoosting,3884.5,45.3,49.8


### Share of listings within ±10% and ±25%, and off by a factor of 2

In [15]:
rows = []
for name in MODELS:
    ratio = out_of_fold_predictions[name] / y_train
    rows.append({
        "Model": name,
        "within ±10% (%)": (errors[SHORT[name] + " abs % error"] <= 10).mean() * 100,
        "within ±25% (%)": (errors[SHORT[name] + " abs % error"] <= 25).mean() * 100,
        "off by a factor ≥ 2 (%)": ((ratio >= 2) | (ratio <= 0.5)).mean() * 100,
    })
pd.DataFrame(rows).set_index("Model").round(1)

,within ±10% (%),within ±25% (%),off by a factor ≥ 2 (%)
Model,,,
Ridge,25.5,56.2,8.8
Random Forest,28.2,60.2,7.5
HistGradientBoosting,32.0,65.3,5.8


## 7. Validation checks

Each check is an `assert`.

In [16]:
# 1. The Ridge reference reproduces notebook 23 (asserted in section 5)
print("1. Ridge reproduces notebook 23 (asserted in section 5).")

# 2. Identical folds for all models, and the same as a fresh GroupKFold(random_state=42)
fresh_splits = list(GroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
                    .split(model_features, y_train, groups=groups))
for name, checks in fold_checks.items():
    for check, (fresh_train, fresh_validation) in zip(checks, fresh_splits):
        assert np.array_equal(check["train_positions"], fresh_train), name
        assert np.array_equal(check["validation_positions"], fresh_validation), name
print("2. All three models used exactly the same 5 folds (= a fresh GroupKFold with random_state=42).")

# 3. No property group crosses a fold boundary
for fold_number, (train_positions, validation_positions) in enumerate(cv_splits, start=1):
    assert set(groups.iloc[train_positions]).isdisjoint(set(groups.iloc[validation_positions])), fold_number
print("3. No property_group_id on both sides of any fold.")

# 4. The test set never entered training or CV
assert set(train["property_id"]).isdisjoint(test_property_ids)
assert (train["split"] == "train").all() and len(model_features) == 16219
print("4. None of the", test_row_count, "test listings was used.")

# 5. Preprocessing fitted separately within each fold (for every model)
for name, checks in fold_checks.items():
    for check in checks:
        assert np.allclose(check["imputer_medians"], check["expected_medians"]), name
    assert not all(np.allclose(checks[0]["imputer_medians"], check["imputer_medians"]) for check in checks[1:]), name
print("5. For every model, each fold's imputer medians = that fold's training medians, and they differ between folds.")
print("   Rare-locality counts per fold:", {name: [check["rare_localities"] for check in checks]
                                             for name, checks in fold_checks.items()})

# 6. One prediction per validation row; every training row validated exactly once
validation_counts = np.zeros(len(model_features), dtype=int)
for train_positions, validation_positions in cv_splits:
    validation_counts[validation_positions] += 1
assert (validation_counts == 1).all()
for name, checks in fold_checks.items():
    for check in checks:
        assert check["prediction_count"] == len(check["validation_positions"]), name
print("6. One prediction per validation row; every training row validated exactly once.")

# 7. No NaN / infinite / non-positive predictions
for name, predictions in out_of_fold_predictions.items():
    assert predictions.notna().all() and np.isfinite(predictions).all() and (predictions > 0).all(), name
    assert predictions.index.equals(model_features.index), name
print("7. No NaN, infinite or non-positive predictions for any model.")

# 8. No target-derived feature
FORBIDDEN_COLUMNS = ["price", "price_band", "price_per_sqft", "loc_avg_price_sqft", "loc_avg_price_lmt",
                     "loc_price_quarter_prev", "loc_price_quarter_current", "loc_growth_quarterly_pct",
                     "loc_locality_rank", "locality_price_encoding", "latitude", "longitude",
                     "locality_centre_lat", "locality_centre_long"]
for column in FORBIDDEN_COLUMNS:
    assert column not in model_features.columns, column
for column in FORBIDDEN_COLUMNS[:9]:
    assert column not in INPUT_COLUMNS, column
print("8. No target-derived column, no locality target encoding and no coordinates among the features.")

# 9. Source files unchanged
for path, md5 in EXPECTED_MD5.items():
    assert file_md5(path) == md5, f"{path.name} changed!"
print("9. cleaned_listings.csv, train_test_split.csv, localities.csv and build_features.py are unchanged.")

1. Ridge reproduces notebook 23 (asserted in section 5).
2. All three models used exactly the same 5 folds (= a fresh GroupKFold with random_state=42).
3. No property_group_id on both sides of any fold.
4. None of the 4081 test listings was used.
5. For every model, each fold's imputer medians = that fold's training medians, and they differ between folds.
   Rare-locality counts per fold: {'Ridge': [1472, 1478, 1472, 1476, 1474], 'Random Forest': [1472, 1478, 1472, 1476, 1474], 'HistGradientBoosting': [1472, 1478, 1472, 1476, 1474]}
6. One prediction per validation row; every training row validated exactly once.
7. No NaN, infinite or non-positive predictions for any model.
8. No target-derived column, no locality target encoding and no coordinates among the features.
9. cleaned_listings.csv, train_test_split.csv, localities.csv and build_features.py are unchanged.


## Model Comparison Findings

All results are 5-fold `GroupKFold` cross-validation on the 16,219 training listings. All three models used the same
folds, the same `build_model_features()` features and the same preprocessed matrix, fitted inside each fold. Ridge
reproduced notebook 23 exactly, every check in section 7 passed, and the test set was not used. No model was tuned.

| Model | MAE (₹ lakh) | Median APE | RMSE (₹ lakh) | R² | Folds better than Ridge (MAE / APE / RMSE / R²) |
|---|---|---|---|---|---|
| Ridge (reference) | 69.37 ± 3.20 | 21.37 ± 0.43% | 177.69 ± 20.73 | 0.491 ± 0.029 | – |
| Random Forest | 66.22 ± 3.68 | 19.52 ± 0.62% | 193.03 ± 27.12 | 0.402 ± 0.052 | 5 / 5 / 0 / 0 |
| HistGradientBoosting | 58.35 ± 2.83 | 17.04 ± 0.42% | 164.76 ± 15.77 | 0.561 ± 0.026 | 5 / 5 / 5 / 5 |

### Did the distance features change the Random Forest's behaviour?

Only partly. Compared with notebook 20 (same configuration, same folds, old lat/long features), the Random Forest
improved on every metric: MAE ₹68.6 → ₹66.2 lakh, Median APE 20.2% → 19.5%, RMSE ₹199.4 → ₹193.0 lakh, R² 0.36 → 0.40.
The **pattern is unchanged**, though:

- it is still better than Ridge for **typical** listings (lower MAE and Median APE in all 5 folds), mostly in the
  ₹1.2–3 Cr middle of the market;
- it still **under-predicts expensive homes badly**: in ₹5 Cr+ the median prediction is 46.5% too low (Ridge −34.5%),
  73% of ₹5 Cr+ listings are predicted below ₹5 Cr, and its largest prediction in the whole training set is only ₹11 Cr
  (the most expensive listing is ₹55 Cr). For ₹20 Cr+ homes it is −81% (notebook 20: −84%);
- so its **RMSE and R² remain worse than Ridge's in every fold**. 91% of its squared error comes from the ₹5 Cr+ band.

The distances helped the forest a little, but they did not fix its ceiling. A forest averages training prices inside
its leaves, and expensive homes share leaves with cheaper ones.

### HistGradientBoosting

With untuned default settings, HGB is **better than Ridge on all four metrics in all five folds**: MAE −₹11.0 lakh
(−15.9%), Median APE −4.3 points, RMSE −₹12.9 lakh, R² +0.07. Its fold-to-fold spread is also no larger than Ridge's (MAE
std ₹2.8 vs ₹3.2 lakh). It also has the most listings within ±10% (32% vs 25.5% for Ridge) and within ±25% (65% vs 56%),
and the fewest off by a factor of 2 or more (5.8% vs 8.8%).

Unlike the forest, it does **not** trade the expensive end for the typical listing:

- ₹5 Cr+: MAE ₹377 lakh vs ₹421 (Ridge) and ₹497 (RF); median −28% vs −35% and −47%. It is the only model that
  predicts more than half of the ₹5 Cr+ listings above ₹5 Cr (55%);
- ₹0–50 L: Median APE 26.9% vs 50.2% (Ridge) and 53.8% (RF). The strong over-prediction of the cheapest homes is roughly
  halved (median +26% vs +50%), although it is still there.

### The trade-offs

**Ridge**
- Strengths: simple and fully explainable (one coefficient per column); the fastest; it can extrapolate along linear
  trends; the smallest errors in ₹80 L–1.2 Cr (Median APE 15.4%, best of the three).
- Weaknesses: the clearest "pull toward the middle" of the three (cheap homes +50%, ₹5 Cr+ −35%); it cannot learn
  interactions; clearly the worst Median APE and MAE here.

**Random Forest**
- Strengths: better than Ridge for typical listings (Median APE 19.5%); in ₹1.5–2 Cr it has the lowest MAE of all three
  (₹30.2 and ₹34.3 lakh).
- Weaknesses: a hard ceiling on predictions, so expensive homes are strongly under-predicted and RMSE/R² are the worst of
  the three; the least stable across folds (R² std 0.052); Penthouses (34 listings) and Builder Floor Apartments are
  over-predicted (+40% and +9%).

**HistGradientBoosting**
- Strengths: the best overall on every metric and in every fold; better than Ridge at both ends of the price range;
  the best Median APE for apartments (13.3%), houses (25.2%) and villas (22.4%).
- Weaknesses:
  - untuned, and with no internal early stopping, so these numbers are one fixed configuration;
  - harder to explain than Ridge;
  - slightly worse than Ridge in ₹80 L–1.2 Cr (Median APE 16.2% vs 15.4%) and slightly worse than the Random Forest in
    ₹1.75–3 Cr MAE;
  - worse than Ridge for Penthouses (28% vs 18%, but only 34 listings).

**Problems that no model solves:**
- **₹20 Cr+ homes** remain far under-predicted by all three (median −64% for HGB, −71% for Ridge, −81% for RF, 48
  listings).
- **Houses and villas** remain about twice as hard as apartments for every model (HGB: 24.3% vs 13.6%).
- The **cheapest listings** are still over-predicted, including the ≤ ₹2 lakh probable entry errors recorded in
  notebook 20.

**No final model is chosen here.**
- HGB is clearly the strongest candidate on this evidence. But it has not been tuned, the other models have not been
  tuned either, and the choice of primary metric is still open.
- Ridge stays the simple, explainable reference. The Random Forest's strength in the middle of the market does not make
  up for its problem at the top.
- All numbers describe **asking** prices in a non-random sample (price-range searches) and come from cross-validation,
  not from the untouched test set.

## Next Step

1. **Hyperparameter tuning of the strongest candidates**, at least HistGradientBoosting and the Ridge reference (the
   Random Forest only if there is a specific reason, since its ceiling problem is structural):
   - use the **same group-aware CV**: `GroupKFold(n_splits=5, shuffle=True, random_state=42)`, `groups =
     property_group_id`, training set only;
   - write down a small, fixed search space **before** running it;
   - fit all preprocessing inside each fold;
   - any early stopping or inner validation must also be group-aware.

   Because the tuning then chooses settings using these folds, the tuned CV scores will be slightly optimistic. The final
   check has to come from data that tuning never saw.
2. **Select the primary evaluation metric** (rupee MAE, Median APE or RMSE) and write it down **before** the final
   comparison. The models rank differently on RMSE than on MAE or Median APE, so the metric must be decided first, not
   chosen after seeing which model wins it.
3. Only then, **one final evaluation on the untouched test set** (4,081 listings) for the selected model(s), with
   no further changes afterwards.